# Avance 5 · Los ensambles de la permeabilidad con huellas

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Compañero del [05](05_Modelo_Final.ipynb) y del [03c](03c_Permeabilidad_huellas.ipynb). Responde una sola pregunta, la que ninguno de los dos contesta: **¿un ensamble saca de las huellas moleculares algo que un modelo individual no saca?**

El [03c](03c_Permeabilidad_huellas.ipynb) midió cinco representaciones sobre los seis modelos **individuales** de la fase 1 y ninguna mejoró a nadie, que es lo que dejó escrito la decisión [1.3](../docs/decisiones.md#d1-3): las huellas entran en la fase 2 y no en la fase 1. El [05](05_Modelo_Final.ipynb) midió los cinco **ensambles**, pero solo con descriptores. El cruce —ensambles **y** huellas— no estaba medido, y la razón para medirlo no es la esperanza de que gane: un ensamble combina modelos que se equivocan distinto, y una representación con 2,048 columnas nuevas podría separarlos más de lo que separa a cada uno por su cuenta. Si no pasa, eso **también** es el resultado, y conviene tenerlo medido en vez de supuesto.

Este cuaderno lleva sufijo de letra, así que **no es un entregable**: es la evidencia detrás de una frase del 05. No elige modelo final, no dibuja los cuatro gráficos y **no toca la partición de prueba**, que se lee una sola vez y se lee en el 05.

## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"
DATOS = RAIZ / "notebooks" / "datos"

from src.evaluation.cv import evaluar
from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.models import cargar, seleccion
from src.models.ensambles_clasificacion import CanalizacionConGrupos, ensambles

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

### Las huellas, y por qué envolver un ensamble no es gratis

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura —el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4— y un 1 si la molécula la tiene. De todas las que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto; es el argumento de explicabilidad que pide la [C.2](../docs/decisiones.md#dc-2).

**El vocabulario se aprende en cada ajuste, con las filas de entrenamiento del pliegue.** Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten, así que calcularlo una vez sobre todo el conjunto dejaría que la validación influyera en qué columnas existen. `AgregarHuellas` va dentro de la canalización por eso, y `src/features/huellas.py` lo explica con detalle.

Hay un matiz que este cuaderno sí introduce y conviene decirlo: el `stacking` parte por dentro, y sus pliegues internos **heredan** el vocabulario elegido con todas las filas de entrenamiento del pliegue externo, no vuelven a elegirlo. No se mira la etiqueta para elegirlo —`AgregarHuellas.fit` solo cuenta frecuencias—, así que no es una fuga de la etiqueta, pero la frase honesta es que el meta-modelo ve columnas escogidas con filas que sus bases usaron para validar. La evaluación externa no se ve afectada.

Y envolver un ensamble en una canalización rompería algo que no se ve:

```python
class CanalizacionConGrupos(Pipeline):
    @property
    def necesita_grupos(self) -> bool:
        return bool(getattr(self.steps[-1][1], "necesita_grupos", False))

    def fit(self, X, y=None, grupos=None, **kw):
        if grupos is not None:
            kw[f"{self.steps[-1][0]}__grupos"] = grupos
        return super().fit(X, y, **kw)
```

El arnés mira `necesita_grupos` **en el objeto que recibe** y, si está puesto, llama `fit(X, y, grupos=...)`. Un `Pipeline` corriente no declara ese atributo ni acepta ese argumento, así que ponerle un `AgregarHuellas` delante al `stacking` con `make_pipeline` haría que el arnés dejara de pasarle los grupos: `StackingAgrupado` reventaría —porque se niega a ajustar sin ellos, y eso es a propósito— o, si no se negara, partiría al azar por dentro y la fuga no aparecería en ninguna métrica. La subclase hereda la exigencia del último paso y le reenvía el argumento.

### Los datos

Entrenamiento y validación de la fase 1, con la prueba reservada fuera. Los mismos compuestos, los mismos grupos por andamio y esqueleto de conectividad, las mismas tres semillas y los mismos cinco pliegues que el 03, el 04 y el 05, así que las filas de este cuaderno se pueden poner al lado de las de aquellos.

Cada compuesto lleva su **origen** —natural exacto, análogo estereoquímico o sin equivalente natural—, que es el desglose de la decisión [1.6](../docs/decisiones.md#d1-6): un promedio de todos no dice cómo le va al modelo con los productos naturales, que son los que el proyecto quiere priorizar.

In [ ]:
t = cargar("permeabilidad", estratos="origen")
y, g, origen = t.y, t.grupos, t.estratos
X_control = t.X                                  # la matriz de descriptores del 03, el 04 y el 05
smiles = t.meta["SMILES"].to_numpy()
assert len(smiles) == len(y) == X_control.shape[0]

N_CONTROL = X_control.shape[1]
print(f"{len(y):,} compuestos · {y.mean():.1%} BBB+ · {len(np.unique(g)):,} grupos · "
      f"{N_CONTROL} descriptores")
print(pd.Series(origen).value_counts().to_string())

### Las huellas de los compuestos

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada ajuste. Antes se agrupan las subestructuras que valen lo mismo en todos los compuestos —porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo—: se queda una sola columna y se guarda el nombre de todas las que representa. Eso no mira la etiqueta y lo idéntico en todas las filas lo es en cualquier subconjunto, así que hacerlo una vez no filtra nada.

Las columnas de descriptores van primero y las huellas después, en una sola matriz dispersa sin densificar.

In [ ]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(smiles)
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
X_mixta = sparse.hstack([sparse.csr_matrix(X_control), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} compuestos; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")
print(f"X_mixta {X_mixta.shape} · cada modelo verá {N_CONTROL} + 2,048 columnas")

## 2. Dos brazos, los mismos nueve modelos

Los nueve son los cuatro individuales del 04 que el 05 usó como referencia y como bases —el SVM RBF, la logística, la red de una capa y los vecinos— y los cinco ensambles del 05, **sin tocar su definición**: vienen de `src/models/ensambles_clasificacion.py` tal cual, para que su fila con descriptores reproduzca la que el 05 registró.

**Dos brazos y no cinco.** El 03c ya midió `huellas`, `seleccionados` y `depurados` sobre los individuales, y ninguno mejoró a nadie; repetirlos aquí contestaría una pregunta que ya tiene respuesta. Los dos que quedan son los que contestan esta: `descriptores`, que es el control y además reproduce el 05, y `descriptores+huellas`. Añadir un tercer brazo es una línea en `BRAZOS`.

**Los afinados quedan fuera.** El 05 midió el bosque y el boosting con búsqueda de rejilla y ninguno de los dos pagó su costo. Este cuaderno no pregunta por el afinado.

### Lo que se fija antes de ver los resultados

- **Las métricas principales son el MCC y el AUPRC de la clase minoritaria** ([C.1](../docs/decisiones.md#dc-1)); el AUROC se reporta como tercera lectura y no decide.
- **Las huellas mejoran a un modelo solo si** la diferencia media contra su propio brazo `descriptores` supera el umbral **y** es positiva en al menos **12 de las 15** evaluaciones, en las dos métricas principales a la vez. Las diferencias se calculan **en la misma evaluación** —misma semilla, mismo pliegue, mismo modelo—, no entre promedios.
- **El umbral de cada métrica es la mayor desviación que esa métrica muestra entre las 15 evaluaciones del control**, calculado solo con el brazo `descriptores` antes de mirar el otro. Es la misma regla con la que se leyeron el 03c, el 04c y el 05 ([1.3](../docs/decisiones.md#d1-3)), y se usa aquí por consistencia.
- Tres advertencias que acompañan a la regla siempre que se cite: **no está pre-registrada**, las 15 evaluaciones **no son independientes** porque las semillas comparten filas, y en la fase 1 el umbral sale **conservador** —la dispersión del MCC entre pliegues agrupados es grande—, así que se reporta como tal en vez de bajarlo para que algo pase.

In [ ]:
SEMILLAS = (0, 1, 2)
N_BITS = 2048                                    # tamaño del vocabulario de Sort & Slice
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
MUESTRA_ENTRENAMIENTO = 3000                     # filas de entrenamiento para medir la brecha
PLIEGUES_BRECHA = 2                              # la brecha es un diagnóstico, no una cifra que decida
BASE = ("svm_rbf", "logistica", "mlp", "knn")     # los cuatro del 04, bases de los heterogéneos
MEJOR_INDIVIDUAL = "svm_rbf"                      # el que eligió el 04
PRINCIPALES = ("mcc", "auprc_minoritaria")        # las que deciden; el AUROC no

individuales = seleccion(BASE)
MODELOS = {**individuales, **ensambles(BASE)}
ORDEN_MODELOS = list(BASE) + ["bosque", "boosting", "bagging_svm", "votacion", "stacking"]
assert set(MODELOS) == set(ORDEN_MODELOS)


def con_huellas(m):
    """El mismo modelo, con el vocabulario aprendido dentro del ajuste y los grupos intactos."""
    return CanalizacionConGrupos([("huellas", AgregarHuellas(N_CONTROL, n_bits=N_BITS)),
                                  ("modelo", m)])


BRAZOS = {
    "descriptores":         (X_control, lambda m: m),
    "descriptores+huellas": (X_mixta,   con_huellas),
}
orden_brazos = list(BRAZOS)
print(f"{len(MODELOS)} modelos × {len(BRAZOS)} brazos × {len(SEMILLAS)} semillas × 5 pliegues")

In [ ]:
partes = []
inicio = time.perf_counter()
for brazo, (matriz, armar) in BRAZOS.items():
    pipes = {nombre: armar(m) for nombre, m in MODELOS.items()}
    r, _ = evaluar(pipes, matriz, y, g, estratos=origen, semillas=SEMILLAS, verboso=False)
    partes.append(r.assign(brazo=brazo))
    print(f"{brazo:<22} {(time.perf_counter() - inicio) / 60:6.1f} min", flush=True)
res = pd.concat(partes, ignore_index=True)
print(f"{len(res):,} filas")

## 3. Resultados

### El control: el brazo `descriptores` repite el 05

Antes de leer nada sobre las huellas hay que comprobar que el brazo de control reproduce lo que ya está registrado. Se compara contra la corrida del 05 **de esta misma máquina** —`2026-10-29-ingrid-1.5-ensambles-permeabilidad`—, no contra la de macOS que trae el cuaderno: la matriz de descriptores de la fase 1 no es idéntica entre las máquinas del equipo ([#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166)) y comparar contra otra máquina mezclaría esa diferencia con cualquier otra.

Si una fila no reproduce, el problema está en este cuaderno y no en las huellas.

In [ ]:
REGISTRO = RAIZ / "experimentos" / "2026-10-29-ingrid-1.5-ensambles-permeabilidad" / "metricas.json"
registrado = pd.DataFrame(json.loads(REGISTRO.read_text())["comparativa"]).T
aqui = (res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
        .groupby("modelo")[["mcc", "auprc_minoritaria", "auroc"]].mean())
control = pd.concat({"05 (registrado)": registrado[["mcc", "auprc_minoritaria", "auroc"]],
                     "05c (esta corrida)": aqui}, axis=1).loc[ORDEN_MODELOS]
for m in ("mcc", "auprc_minoritaria", "auroc"):
    control[("diferencia", m)] = control[("05c (esta corrida)", m)] - control[("05 (registrado)", m)]
display(control.round(4))
print(f"filas que reproducen a tres decimales: "
      f"{int((control['diferencia'].abs() < 5e-4).all(axis=1).sum())} de {len(control)}")

### Los dos brazos, modelo por modelo

In [ ]:
def tabla(r, estrato="todos"):
    """Media por brazo y modelo de cada métrica, con la dispersión del MCC entre evaluaciones."""
    r = r[r["estrato"] == estrato]
    t = r.groupby(["brazo", "modelo"])[["mcc", "auprc_minoritaria", "auroc", "segundos_ajuste"]].mean()
    t.insert(1, "desv_mcc", r.groupby(["brazo", "modelo"])["mcc"].std())
    return t.round(4)


tabla_todos = tabla(res)
for m in ("mcc", "auprc_minoritaria", "auroc"):
    print(m)
    display(tabla_todos[m].unstack("brazo")[orden_brazos].loc[ORDEN_MODELOS])

### ¿Mejoran las huellas? La comparación pliegue por pliegue

Cada modelo contra **sí mismo** con descriptores, en la misma semilla y el mismo pliegue. `mejor_en` cuenta en cuántas de las 15 gana; `cumple` aplica la regla a esa métrica y `mejora` exige las dos principales a la vez. Los umbrales se calculan primero, solo con el control.

In [ ]:
control_todos = res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")]
UMBRALES = {m: round(float(control_todos.groupby("modelo")[m].std().max()), 3) for m in PRINCIPALES}
print("Umbrales: la mayor desviación de cada métrica del control entre sus 15 evaluaciones")
print(UMBRALES)


def comparacion(r, estrato="todos"):
    """Diferencia de cada brazo contra `descriptores`, evaluación por evaluación."""
    r = r[r["estrato"] == estrato]
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["modelo", "semilla", "pliegue"], columns="brazo", values=metrica)
        for brazo in orden_brazos[1:]:
            dif = (p[brazo] - p["descriptores"]).dropna()
            for modelo, v in dif.groupby("modelo"):
                filas.append({"brazo": brazo, "modelo": modelo, "metrica": metrica,
                              "evaluaciones": len(v), "dif_media": v.mean(), "dif_desv": v.std(),
                              "mejor_en": int((v > 0).sum())})
    c = pd.DataFrame(filas).set_index(["brazo", "modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(UMBRALES).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("mejora")
    return c.round(4), mejora


comparacion_todos, mejora_todos = comparacion(res)
display(comparacion_todos.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
mejora_todos.unstack("brazo")[orden_brazos[1:]].loc[ORDEN_MODELOS]

### Con huellas, ¿algún ensamble supera al mejor individual?

La otra lectura, la del 05: dentro del brazo con huellas, cada ensamble contra el `svm_rbf` **medido en el mismo brazo y la misma evaluación**. Es la pregunta de la rúbrica —si el ensamble vale la pena— hecha sobre la representación nueva, y usa la misma regla con el umbral recalculado sobre el control de ese brazo.

In [ ]:
def contra_individual(r, brazo, estrato="todos"):
    """Cada modelo contra el mejor individual del mismo brazo, evaluación por evaluación."""
    r = r[(r["brazo"] == brazo) & (r["estrato"] == estrato)]
    umbrales = {m: round(float(r[r["modelo"].eq(MEJOR_INDIVIDUAL)][m].std()), 3) for m in PRINCIPALES}
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["semilla", "pliegue"], columns="modelo", values=metrica)
        for modelo in [m for m in ORDEN_MODELOS if m != MEJOR_INDIVIDUAL]:
            dif = (p[modelo] - p[MEJOR_INDIVIDUAL]).dropna()
            filas.append({"modelo": modelo, "metrica": metrica, "evaluaciones": len(dif),
                          "dif_media": dif.mean(), "dif_desv": dif.std(),
                          "mejor_en": int((dif > 0).sum())})
    c = pd.DataFrame(filas).set_index(["modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(umbrales).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("supera")
    return umbrales, c.round(4), mejora


for brazo in orden_brazos:
    umbrales_b, contra, supera = contra_individual(res, brazo)
    print(f"{brazo} · umbrales del {MEJOR_INDIVIDUAL} en este brazo: {umbrales_b}")
    display(contra.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
    display(supera.to_frame().T)

### Por origen

Los umbrales se calcularon con el agregado, así que estas filas son **exploratorias**: sirven para ver dónde cae la diferencia, no para declarar una mejora. Importa porque en la fase 2 la ganancia de las huellas se concentró en los pares que solo tiene ChEMBL y no en los productos naturales, y aquí hay que mirar si pasa lo mismo.

In [ ]:
por_origen = list(pd.unique(res.loc[res["estrato"].ne("todos"), "estrato"]))
mcc_origen = pd.concat({e: tabla(res, e)["mcc"].unstack("brazo")[orden_brazos] for e in por_origen},
                       axis=1)
display(mcc_origen.loc[ORDEN_MODELOS].round(4))
pd.concat({e: comparacion(res, e)[0].query("metrica in @PRINCIPALES")[["dif_media", "mejor_en"]]
           for e in por_origen}, names=["origen"]).round(3)

### La brecha entre entrenamiento y validación

2,048 columnas nuevas sobre 6,306 compuestos invitan a memorizar, y el promedio no lo muestra. En la fase 2 la red con huellas llegó a 0.252 de brecha y fue el único modelo que empeoró. Se mide con el AUROC sobre una muestra de las filas de entrenamiento y sobre el pliegue de validación, con una semilla y los dos primeros pliegues, porque la brecha es un diagnóstico y no una cifra que decida: ninguna conclusión de este cuaderno se apoya en ella, y volver a ajustar los nueve modelos de los dos brazos en los cinco pliegues costaría más que la comparativa entera.

In [ ]:
filas_brecha = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
for k, (ajuste, val) in enumerate(list(cv.split(X_control, y, g))[:PLIEGUES_BRECHA], 1):
    muestra = np.random.default_rng(k).choice(ajuste, min(MUESTRA_ENTRENAMIENTO, len(ajuste)),
                                              replace=False)
    for brazo, (matriz, armar) in BRAZOS.items():
        for nombre, base in MODELOS.items():
            m = armar(base)
            if getattr(m, "necesita_grupos", False):
                m.fit(matriz[ajuste], y[ajuste], grupos=g[ajuste])
            else:
                m.fit(matriz[ajuste], y[ajuste])
            filas_brecha.append({
                "brazo": brazo, "modelo": nombre, "pliegue": k,
                "auroc_entrenamiento": roc_auc_score(y[muestra],
                                                     m.predict_proba(matriz[muestra])[:, 1]),
                "auroc_validacion": roc_auc_score(y[val], m.predict_proba(matriz[val])[:, 1])})
brecha = pd.DataFrame(filas_brecha).groupby(["brazo", "modelo"])[
    ["auroc_entrenamiento", "auroc_validacion"]].mean()
brecha["brecha"] = brecha["auroc_entrenamiento"] - brecha["auroc_validacion"]
brecha.round(4).unstack("brazo")["brecha"][orden_brazos].loc[ORDEN_MODELOS]

In [ ]:
# Registro en experimentos/ (decisión 1.3: qué representación reciben los modelos).
from src.tracking import Corrida

with Corrida("ensambles-permeabilidad-huellas", decision="1.3",
             params={"modelos": ORDEN_MODELOS, "brazos": list(BRAZOS),
                     "base_heterogeneos": list(BASE), "mejor_individual": MEJOR_INDIVIDUAL,
                     "pliegues": 5, "n_bits_sortslice": N_BITS, "radio_morgan": 2,
                     "vocabulario": "se aprende en cada ajuste; los pliegues internos del stacking lo heredan",
                     "metricas_principales": list(PRINCIPALES),
                     "metrica_tercera_lectura": "auroc",
                     "afinados": "ninguno; el 05 midió que no pagan",
                     "pliegues_brecha": PLIEGUES_BRECHA,
                     "regla_mejora": {"umbral_por_metrica": UMBRALES,
                                      "minimo_evaluaciones": MINIMO_PLIEGUES, "de": 15,
                                      "criterio": "las dos metricas principales a la vez",
                                      "umbral": "mayor desviacion del control entre sus 15 evaluaciones",
                                      "comparacion_decisiva": "agregado contra descriptores",
                                      "estratos": "exploratorios",
                                      "estado": "no pre-registrada; las 15 evaluaciones no son independientes"},
                     "control_registrado": "2026-10-29-ingrid-1.5-ensambles-permeabilidad",
                     "prueba_reservada": "no se toca; se lee una sola vez en el 05"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    metricas = {
        "todos": {f"{b}/{m}": v for (b, m), v in tabla_todos.to_dict(orient="index").items()},
        "comparacion_todos": {f"{b}/{m}/{k}": v for (b, m, k), v in
                              comparacion_todos.to_dict(orient="index").items()},
        "brecha": {f"{b}/{m}": v for (b, m), v in brecha.to_dict(orient="index").items()},
        "control_reproduce": {m: float(control[("diferencia", m)].abs().max())
                              for m in ("mcc", "auprc_minoritaria", "auroc")},
        **{e: {f"{b}/{m}": v for (b, m), v in tabla(res, e).to_dict(orient="index").items()}
           for e in por_origen}}
    for brazo in orden_brazos:
        _, contra, _ = contra_individual(res, brazo)
        metricas[f"contra_individual/{brazo}"] = {f"{m}/{k}": v for (m, k), v in
                                                  contra.to_dict(orient="index").items()}
    c.metricas(metricas)

print(c.carpeta.name)

In [ ]:
# Las 15 evaluaciones por brazo, modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba. El CSV es para rehacer la lectura y las diferencias
# pareadas sin repetir la corrida.
# Va después del registro a propósito: escribirlo antes dejaría el árbol sucio al registrar.
SALIDA = DATOS / "05c_brazos.csv"
res.to_csv(SALIDA, index=False)
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")

## 3.1 Lectura

*La escribe el equipo mirando las tablas de arriba, y no la decide la primera fila. Lo que hay que contestar, dicho antes de ver los resultados para que la lectura no se acomode a ellos:*

1. *¿Alguna fila de `mejora` sale verdadera? Si ninguna, decirlo es el resultado, y hay que decirlo con el umbral a la vista en vez de bajarlo.*
2. *¿Algún ensamble se comporta distinto que los individuales del 03c, es decir, saca de las huellas algo que un modelo solo no saca? La comparación es contra el 03c, no contra cero.*
3. *¿Dónde cae la diferencia por origen? Si vuelve a caer fuera de los productos naturales, como en la fase 2, eso pesa más que el promedio.*
4. *¿Qué dice la brecha? Una ganancia pequeña con una brecha grande no es una ganancia.*
5. *¿Cambia esto algo de la [1.3](../docs/decisiones.md#d1-3)? Si no, la entrada se queda como está y este cuaderno es su evidencia; si sí, hay que marcar la versión superada y decir qué la cambió.*

*Por completar por el equipo.*

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Andamiaje del cuaderno, escrito antes de ver cualquier resultado: `CanalizacionConGrupos` para que envolver un ensamble no corte el paso de los grupos, el armado de los dos brazos reutilizando los ensambles del 05 sin modificarlos, y las dos comparaciones pareadas | Secciones 1 a 3 (el código de apoyo vive en `src/`). La lectura de la sección 3.1 y la tabla de contribución las escribe el equipo, sin IA | **Lo que el equipo decidió:** medir el cruce de ensambles y huellas aunque el 03c y la [1.3](../docs/decisiones.md#d1-3) hicieran esperar que no mejore, para tener la evidencia escrita en vez de supuesta; cuáles dos brazos y por qué no cinco; dejar los afinados fuera; y la lectura de los resultados |

### Decisiones principales

1. **Medir el cruce en vez de declararlo.** La [1.3](../docs/decisiones.md#d1-3) ya dice que las huellas no entran en la fase 1, y el 03c lo midió sobre los individuales. *Alternativa descartada:* dejar la frase «no está medido para ensambles» como límite declarado en el 05, que cuesta cero pero deja una afirmación sin respaldo en un avance que sí tiene cómo respaldarla.
2. **Dos brazos, no cinco.** `descriptores` y `descriptores+huellas`. *Alternativa descartada:* repetir los cinco brazos del 03c, que multiplica la corrida para contestar tres preguntas que ya tienen respuesta en una corrida registrada.
3. **El control se compara contra la corrida de esta misma máquina.** *Alternativa descartada:* contrastarlo con la corrida de macOS que trae el 05, que mezclaría la diferencia entre matrices del [#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166) con cualquier otra diferencia y dejaría sin saber si el control reproduce.

### Contribución individual

*Por completar.*